# 06_Scikit_Learn.ipynb — Polynomial Regression

Unlike other algorithms, **Polynomial Regression does not have its own estimator** in scikit-learn.

This is one of the most important things to remember.

Instead, it uses:

```text
PolynomialFeatures
        +
LinearRegression
```

or

```text
PolynomialFeatures
        +
Ridge
```

or

```text
PolynomialFeatures
        +
Lasso
```

---

# 1. Estimator

There is **no**

```python
PolynomialRegression()
```

class.

Instead we use

```python
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
```

Then

```python
poly = PolynomialFeatures(degree=2)

X_poly = poly.fit_transform(X)

model = LinearRegression()

model.fit(X_poly, y)
```

Notice

`PolynomialFeatures`

is **NOT** a machine learning model.

It is only a **transformer**.

The real estimator is

```python
LinearRegression()
```

---

# 2. Import & Constructor

## Required Imports

```python
from sklearn.preprocessing import PolynomialFeatures

from sklearn.linear_model import LinearRegression

from sklearn.pipeline import Pipeline
```

---

## Constructor Syntax

```python
PolynomialFeatures(
    degree=2,
    interaction_only=False,
    include_bias=True,
    order="C"
)
```

---

## Parameter Table

| Parameter        | Default | Description                                               | Common Usage                                |
| ---------------- | ------- | --------------------------------------------------------- | ------------------------------------------- |
| degree           | 2       | Highest polynomial degree to generate                     | Usually 2 or 3                              |
| interaction_only | False   | Generate only interaction terms, skip squared/cubic terms | Rare                                        |
| include_bias     | True    | Adds a column of ones                                     | Usually False with sklearn LinearRegression |
| order            | 'C'     | Memory layout of output array                             | Leave default                               |

---

## Important Parameters Explained

### 1. degree

Most important parameter.

```python
PolynomialFeatures(degree=2)
```

creates

```text
x

x²
```

Degree 3

creates

```text
x

x²

x³
```

Higher degree

↓

More flexibility

↓

Higher overfitting risk.

---

### 2. interaction_only

Default

```python
False
```

Suppose

```text
Age

Salary
```

Normal output

```text
Age

Salary

Age²

Age Salary

Salary²
```

If

```python
interaction_only=True
```

Output becomes

```text
Age

Salary

Age Salary
```

No

```text
Age²

Salary²
```

Only interaction terms.

Rarely used.

---

### 3. include_bias

Default

```python
True
```

Output

```text
1

Age

Age²
```

The first column is all ones.

Since

```python
LinearRegression()
```

already learns an intercept by default (`fit_intercept=True`), this bias column is usually unnecessary.

**Recommended practice:**

```python
PolynomialFeatures(
    degree=2,
    include_bias=False
)
```

---

### 4. order

Determines how the transformed array is stored in memory (`"C"` row-major or `"F"` column-major).

It **does not change the mathematical result**.

Leave it at the default.

---

# Recommended Settings

```python
PolynomialFeatures(
    degree=2,
    include_bias=False
)
```

Then

```python
LinearRegression()
```

or even better

```python
Ridge()
```

---

# Best Practices

* Start with **degree = 2**
* Avoid very high degrees unless justified
* Scale features when using regularized models (Ridge/Lasso/Elastic Net)
* Prefer a `Pipeline` to avoid data leakage
* Use **cross-validation** to choose the degree
* For high-degree polynomials, consider **Ridge Regression** to reduce overfitting

---

# 3. Methods & Attributes

## Methods Table

| Method                  | Purpose                                                | Returns                |
| ----------------------- | ------------------------------------------------------ | ---------------------- |
| fit(X)                  | Learns how many features to generate (stores metadata) | self                   |
| transform(X)            | Generates polynomial features                          | NumPy array            |
| fit_transform(X)        | Fits and transforms in one step                        | NumPy array            |
| get_feature_names_out() | Returns names of generated features                    | NumPy array of strings |

---

## Attributes Table

| Attribute         | Description                                                            |
| ----------------- | ---------------------------------------------------------------------- |
| powers_           | Exponents applied to each original feature for every generated feature |
| n_features_in_    | Number of original input features                                      |
| feature_names_in_ | Names of input features (if provided)                                  |

---

## Commonly Used Methods

### fit()

```python
poly.fit(X)
```

Learns metadata (e.g., number of input features). It **does not** compute any model weights.

---

### transform()

```python
X_poly = poly.transform(X)
```

Transforms the input into polynomial features.

Use this on **test data** after fitting on the training data.

---

### fit_transform()

```python
X_poly = poly.fit_transform(X)
```

Most common during training.

Equivalent to:

```python
poly.fit(X)
X_poly = poly.transform(X)
```

---

### get_feature_names_out()

Very useful for understanding generated features.

```python
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures

df = pd.DataFrame({
    "Age":[20,30],
    "Salary":[50000,60000]
})

poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

poly.fit(df)

print(poly.get_feature_names_out())
```

Output

```text
['Age'
 'Salary'
 'Age^2'
 'Age Salary'
 'Salary^2']
```

This makes it easy to inspect the transformed feature set.

---

## Commonly Used Attributes

### powers_

```python
print(poly.powers_)
```

Output

```text
[[1 0]
 [0 1]
 [2 0]
 [1 1]
 [0 2]]
```

Meaning:

| Row   | Feature      |
| ----- | ------------ |
| [1 0] | Age          |
| [0 1] | Salary       |
| [2 0] | Age²         |
| [1 1] | Age × Salary |
| [0 2] | Salary²      |

This attribute is mainly useful for understanding how the transformed features were constructed.

---

### n_features_in_

```python
print(poly.n_features_in_)
```

Output

```text
2
```

The transformer received two original input features.

---

# One Complete Code Example

```python
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Dataset
X, y = make_regression(
    n_samples=100,
    n_features=1,
    noise=15,
    random_state=42
)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Polynomial transformation
poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Model
model = LinearRegression()

model.fit(X_train_poly, y_train)

# Prediction
pred = model.predict(X_test_poly)

print("R²:", r2_score(y_test, pred))
```

---

# 4. End-to-End Workflow

## Workflow Diagram

```text
Load Data
    ↓
Train-Test Split
    ↓
Polynomial Feature Generation
    ↓
Train Linear Regression
    ↓
Predict
    ↓
Evaluate
```

---

## One Complete Working Example (Recommended Approach)

Instead of manually transforming the data, use a `Pipeline`.

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

pipeline = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", LinearRegression())
])

pipeline.fit(X_train, y_train)

pred = pipeline.predict(X_test)
```

### Why is this better?

The pipeline automatically:

* Fits `PolynomialFeatures` on the training data.
* Applies the same transformation to the test data.
* Trains the regression model.
* Applies the correct transformation during prediction.

This avoids common mistakes and data leakage.

---

## Important Notes

* `PolynomialFeatures` is a **transformer**, not an estimator.
* Always fit the transformer **only on the training data**.
* Use `transform()` (not `fit_transform()`) on the test set.
* A `Pipeline` is the recommended approach in almost all real-world projects.

---

## Common Errors

| Mistake                                                       | Why It Happens                                                                                           |
| ------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------- |
| Calling `predict(X_test)` without transforming `X_test`       | The model expects polynomial features, not the original features.                                        |
| Using `fit_transform()` on the test set                       | This refits the transformer and can introduce data leakage. Use `transform()` instead.                   |
| Choosing a very high degree                                   | Generates too many features and often leads to overfitting.                                              |
| Forgetting `include_bias=False` when using `LinearRegression` | This creates a redundant column of ones because `LinearRegression` already fits an intercept by default. |
| Not using a `Pipeline`                                        | Manual transformations are easier to get wrong and can lead to inconsistent preprocessing.               |

---

# Summary

Unlike every other regression algorithm you've studied:

* **Polynomial Regression has no dedicated estimator.**
* The estimator is still **LinearRegression**, **Ridge**, **Lasso**, or **ElasticNet**.
* `PolynomialFeatures` is simply a preprocessing step that creates additional features.
* In production code, **`Pipeline(PolynomialFeatures → Regression Model)` is the standard and recommended pattern.**

---

In [2]:
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Dataset
X, y = make_regression(
    n_samples=100,
    n_features=1,
    noise=15,
    random_state=42
)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Polynomial transformation
poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Model
model = LinearRegression()

model.fit(X_train_poly, y_train)

# Prediction
pred = model.predict(X_test_poly)

print("R²:", r2_score(y_test, pred))

R²: 0.8727118951514723
